In [2]:
# ============================================================
# GEN QR — ADVANCED QR CODE GENERATOR
# Python + Flask + SQLite + qrcode + Pillow + ipywidgets
# Designed for Anaconda Jupyter Notebook on Windows
# ============================================================

# CELL 1: INSTALL REQUIRED LIBRARIES
%pip install Flask qrcode[pil] Pillow requests ipywidgets


# CELL 2: IMPORT LIBRARIES
import io
import re
import sqlite3
import threading
import time
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlencode, urlparse

import qrcode
import requests
import ipywidgets as widgets

from flask import Flask, jsonify, request, send_file
from IPython.display import display, Image, HTML


# CELL 3: PROJECT CONFIGURATION
PROJECT_DIR = Path.home() / "GEN_QR_Project"
QR_DIR = PROJECT_DIR / "qr_images"
DB_PATH = PROJECT_DIR / "gen_qr.db"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
QR_DIR.mkdir(parents=True, exist_ok=True)

app = Flask("GEN_QR")
app.config["MAX_CONTENT_LENGTH"] = 1024 * 1024

print("GEN QR initialized!")
print("Project location:", PROJECT_DIR)


# CELL 4: DATABASE
def get_db():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def init_db():
    with get_db() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS qr_history (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                qr_type TEXT NOT NULL,
                label TEXT NOT NULL,
                payload TEXT NOT NULL,
                filename TEXT NOT NULL UNIQUE,
                created_at TEXT NOT NULL
            )
        """)


init_db()


# CELL 5: VALIDATION FUNCTIONS
def valid_url(url):
    try:
        parsed = urlparse(url.strip())
        return (
            parsed.scheme in ("http", "https")
            and bool(parsed.netloc)
            and "." in parsed.netloc
        )
    except Exception:
        return False


def valid_upi(upi):
    pattern = r"^[A-Za-z0-9][A-Za-z0-9._-]{1,255}@[A-Za-z0-9][A-Za-z0-9.-]{1,63}$"
    return bool(re.fullmatch(pattern, upi.strip()))


# CELL 6: QR GENERATION FUNCTIONS
def make_qr(payload):
    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_M,
        box_size=10,
        border=4
    )

    qr.add_data(payload)
    qr.make(fit=True)

    image = qr.make_image(
        fill_color="black",
        back_color="white"
    ).convert("RGB")

    output = io.BytesIO()
    image.save(output, format="PNG")

    return output.getvalue()


def save_qr(qr_type, label, payload):
    image_data = make_qr(payload)

    safe_label = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        label
    )[:40].strip("_") or "GEN_QR"

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    filename = f"{safe_label}_{timestamp}.png"

    image_path = QR_DIR / filename
    image_path.write_bytes(image_data)

    created_at = datetime.now(timezone.utc).isoformat()

    with get_db() as conn:
        cursor = conn.execute("""
            INSERT INTO qr_history
            (qr_type, label, payload, filename, created_at)
            VALUES (?, ?, ?, ?, ?)
        """, (
            qr_type,
            label[:120],
            payload,
            filename,
            created_at
        ))

        record_id = cursor.lastrowid

    return {
        "id": record_id,
        "type": qr_type,
        "label": label,
        "payload": payload,
        "filename": filename,
        "created_at": created_at
    }


# CELL 7: FLASK API
@app.route("/")
def home():
    return jsonify({
        "project": "GEN QR",
        "status": "running",
        "features": [
            "URL QR Generator",
            "UPI QR Generator",
            "QR History",
            "PNG Downloads"
        ],
        "endpoints": {
            "health": "/api/health",
            "url_qr": "/api/qr/url",
            "upi_qr": "/api/qr/upi",
            "history": "/api/history"
        }
    })


@app.route("/api/health")
def health():
    return jsonify({
        "status": "ok",
        "project": "GEN QR"
    })


@app.route("/api/qr/url", methods=["POST"])
def generate_url():
    data = request.get_json(silent=True) or {}

    url = str(data.get("url", "")).strip()
    label = str(data.get("label", "Website QR")).strip()

    if not valid_url(url):
        return jsonify({
            "error": "Enter a valid URL beginning with http:// or https://"
        }), 400

    result = save_qr("url", label or "Website QR", url)

    return jsonify({
        "message": "URL QR generated successfully",
        "qr": result
    }), 201


@app.route("/api/qr/upi", methods=["POST"])
def generate_upi():
    data = request.get_json(silent=True) or {}

    upi_id = str(data.get("upi_id", "")).strip()
    payee = str(data.get("payee_name", "")).strip()
    amount = str(data.get("amount", "")).strip()
    note = str(data.get("note", "")).strip()

    if not valid_upi(upi_id):
        return jsonify({
            "error": "Invalid UPI ID format. Example: name@bank"
        }), 400

    if len(payee) > 80 or len(note) > 100:
        return jsonify({
            "error": "Payee name or note is too long"
        }), 400

    if amount:
        try:
            amount_value = float(amount)

            if amount_value <= 0 or amount_value > 10000000:
                raise ValueError

            amount = f"{amount_value:.2f}"

        except ValueError:
            return jsonify({
                "error": "Enter a valid positive amount"
            }), 400

    params = {
        "pa": upi_id,
        "cu": "INR"
    }

    if payee:
        params["pn"] = payee

    if amount:
        params["am"] = amount

    if note:
        params["tn"] = note

    payload = "upi://pay?" + urlencode(params)

    result = save_qr(
        "upi",
        "UPI_" + (payee or upi_id),
        payload
    )

    return jsonify({
        "message": "UPI QR generated successfully",
        "warning": "Verify recipient and amount before making payment.",
        "qr": result
    }), 201


@app.route("/api/history")
def history():
    with get_db() as conn:
        rows = conn.execute("""
            SELECT *
            FROM qr_history
            ORDER BY id DESC
            LIMIT 100
        """).fetchall()

    return jsonify({
        "count": len(rows),
        "items": [dict(row) for row in rows]
    })


@app.route("/api/qr/<int:record_id>/download")
def download_qr(record_id):
    with get_db() as conn:
        row = conn.execute(
            "SELECT filename FROM qr_history WHERE id=?",
            (record_id,)
        ).fetchone()

    if row is None:
        return jsonify({"error": "QR not found"}), 404

    path = QR_DIR / row["filename"]

    if not path.exists():
        return jsonify({"error": "Image file not found"}), 404

    return send_file(
        path,
        mimetype="image/png",
        as_attachment=True,
        download_name=row["filename"]
    )


@app.route("/api/history/<int:record_id>", methods=["DELETE"])
def delete_history(record_id):
    with get_db() as conn:
        row = conn.execute(
            "SELECT filename FROM qr_history WHERE id=?",
            (record_id,)
        ).fetchone()

        if row is None:
            return jsonify({"error": "Record not found"}), 404

        conn.execute(
            "DELETE FROM qr_history WHERE id=?",
            (record_id,)
        )

    (QR_DIR / row["filename"]).unlink(missing_ok=True)

    return jsonify({"message": "QR history deleted"})


# CELL 8: START FLASK SERVER
# Run this cell only ONCE per kernel session.
def run_server():
    app.run(
        host="127.0.0.1",
        port=5000,
        debug=False,
        use_reloader=False,
        threaded=True
    )


server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()
time.sleep(2)

print("GEN QR server started.")
print("API address: http://127.0.0.1:5000")


# CELL 9: PYTHON NOTEBOOK DASHBOARD
qr_type = widgets.ToggleButtons(
    options=[
        ("Website QR", "url"),
        ("UPI Payment QR", "upi")
    ],
    description="QR Type:"
)

url_box = widgets.Text(
    description="URL:",
    placeholder="https://example.com",
    layout=widgets.Layout(width="75%")
)

label_box = widgets.Text(
    description="Label:",
    value="Website QR",
    layout=widgets.Layout(width="75%")
)

upi_box = widgets.Text(
    description="UPI ID:",
    placeholder="name@bank",
    layout=widgets.Layout(width="75%")
)

payee_box = widgets.Text(
    description="Payee:",
    placeholder="Optional payee name",
    layout=widgets.Layout(width="75%")
)

amount_box = widgets.Text(
    description="Amount:",
    placeholder="Optional INR amount",
    layout=widgets.Layout(width="75%")
)

note_box = widgets.Text(
    description="Note:",
    placeholder="Optional payment note",
    layout=widgets.Layout(width="75%")
)

generate_btn = widgets.Button(
    description="Generate QR",
    button_style="success",
    icon="check"
)

history_btn = widgets.Button(
    description="Refresh History",
    icon="refresh"
)

delete_id_box = widgets.IntText(
    description="Delete ID:",
    value=0
)

delete_btn = widgets.Button(
    description="Delete Record",
    button_style="danger"
)

output = widgets.Output()
history_output = widgets.Output()


def update_fields(change=None):
    is_url = qr_type.value == "url"

    url_box.layout.display = "" if is_url else "none"
    label_box.layout.display = "" if is_url else "none"

    upi_box.layout.display = "" if not is_url else "none"
    payee_box.layout.display = "" if not is_url else "none"
    amount_box.layout.display = "" if not is_url else "none"
    note_box.layout.display = "" if not is_url else "none"


qr_type.observe(update_fields, names="value")
update_fields()


def generate_clicked(button):
    with output:
        output.clear_output()

        try:
            if qr_type.value == "url":
                response = requests.post(
                    "http://127.0.0.1:5000/api/qr/url",
                    json={
                        "url": url_box.value.strip(),
                        "label": label_box.value.strip() or "Website QR"
                    },
                    timeout=10
                )

            else:
                response = requests.post(
                    "http://127.0.0.1:5000/api/qr/upi",
                    json={
                        "upi_id": upi_box.value.strip(),
                        "payee_name": payee_box.value.strip(),
                        "amount": amount_box.value.strip(),
                        "note": note_box.value.strip()
                    },
                    timeout=10
                )

            data = response.json()

            if response.status_code != 201:
                print("Error:", data.get("error", data))
                return

            qr = data["qr"]
            image_path = QR_DIR / qr["filename"]

            print(data["message"])
            print("History ID:", qr["id"])
            print("Label:", qr["label"])
            print("QR data:", qr["payload"])

            display(Image(filename=str(image_path)))

            print("Saved to:", image_path.resolve())
            print(
                "Download API:",
                f"http://127.0.0.1:5000/api/qr/{qr['id']}/download"
            )

            if qr["type"] == "upi":
                print(
                    "Safety: QR generation does not verify the UPI account "
                    "or complete a payment. Verify details before paying."
                )

        except Exception as error:
            print("Error:", error)
            print("Make sure the Flask server is running.")


def history_clicked(button):
    with history_output:
        history_output.clear_output()

        try:
            response = requests.get(
                "http://127.0.0.1:5000/api/history",
                timeout=5
            )

            items = response.json()["items"]

            if not items:
                print("No QR history yet.")
                return

            for item in items:
                print(
                    f"ID: {item['id']} | "
                    f"Type: {item['qr_type'].upper()} | "
                    f"Label: {item['label']} | "
                    f"Created: {item['created_at']}"
                )

                image_path = QR_DIR / item["filename"]

                if image_path.exists():
                    display(Image(
                        filename=str(image_path),
                        width=150
                    ))

                print("-" * 60)

        except Exception as error:
            print("Could not load history:", error)


def delete_clicked(button):
    with output:
        output.clear_output()

        record_id = delete_id_box.value

        if record_id <= 0:
            print("Enter a valid history ID.")
            return

        try:
            response = requests.delete(
                f"http://127.0.0.1:5000/api/history/{record_id}",
                timeout=5
            )

            print(response.json())

        except Exception as error:
            print("Delete failed:", error)


generate_btn.on_click(generate_clicked)
history_btn.on_click(history_clicked)
delete_btn.on_click(delete_clicked)


dashboard = widgets.VBox([
    widgets.HTML(
        "<h1>GEN QR</h1>"
        "<h3>Advanced QR Code Generator</h3>"
        "<p>Generate website and UPI QR codes, preview images, "
        "download PNGs and manage history.</p>"
    ),
    qr_type,
    url_box,
    label_box,
    upi_box,
    payee_box,
    amount_box,
    note_box,
    widgets.HBox([generate_btn, history_btn]),
    output,
    widgets.HTML("<h3>Delete a history record</h3>"),
    widgets.HBox([delete_id_box, delete_btn]),
    widgets.HTML("<h3>QR History</h3>"),
    history_output
])

display(dashboard)

print("Dashboard ready!")


# CELL 10: OPTIONAL API TEST
# Run this cell whenever you want to check server health.
try:
    print(
        requests.get(
            "http://127.0.0.1:5000/api/health",
            timeout=3
        ).json()
    )
except Exception as error:
    print("Server check failed:", error)


Note: you may need to restart the kernel to use updated packages.
GEN QR initialized!
Project location: C:\Users\Dhruv Maurya\GEN_QR_Project
 * Serving Flask app 'GEN_QR'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit


GEN QR server started.
API address: http://127.0.0.1:5000


Dashboard ready!


127.0.0.1 - - [27/Oct/2026 13:50:38] "GET /api/health HTTP/1.1" 200 -


{'project': 'GEN QR', 'status': 'ok'}


127.0.0.1 - - [27/Oct/2026 13:53:32] "POST /api/qr/url HTTP/1.1" 201 -
